# Evaluate an evolved policy on ONE brain group per run — TEST (held-out) or TRAIN (sanity)

Companion to `compare_proofreader_policy.ipynb`, for a run launched with
`--train-brains … --test-brains …`. It runs the run's final accepted policy on a
SINGLE brain group per kernel session, selected by the `TRAIN_OR_TEST` parameter:

* `TRAIN_OR_TEST = "test"` — each **TEST** brain, scored on its held-out neurons.
  A WHOLE brain the reviser never saw: a clean held-out generalization measurement.
  **This is the real result.**
* `TRAIN_OR_TEST = "train"` — each **TRAIN** brain, scored on the TRAIN neurons the
  reviser got feedback on. A **SANITY CHECK only**: the policy was evolved against
  these neurons, so high split-repair coverage is EXPECTED — use it to confirm the
  policy fires on the brain it was tuned on and to size the train→test gap, NOT as
  evidence of generalization.

**Why one group per run:** each brain's fragment graph is large (millions of nodes),
and loading more than one in a single kernel session is what exhausts memory. So this
notebook loads + scores exactly ONE group, then frees the graph. To get BOTH: run with
`TRAIN_OR_TEST="test"`, then **restart the kernel**, set `TRAIN_OR_TEST="train"`, and
run again. The two runs write separate output files (suffixed by the group), so nothing
is overwritten.

It auto-detects the test + train brains from the run's `split.json` (`cross_brain` /
`test_brains` / `train_brains`) and whether the run was `splits_only` from the ledger,
so you only set `RUN_NAME` and `TRAIN_OR_TEST`. Falls back gracefully to a per-brain
run's held-out / train subsets if the run was NOT cross-brain (with a warning).

## 0. Setup — env, imports, parameters (RUN_NAME + TRAIN_OR_TEST), auto-detect brains

In [1]:
import os, sys, json, gc
from pathlib import Path
from collections import defaultdict

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Same env / path setup as compare_proofreader_policy.ipynb.
os.environ["GOOGLE_APPLICATION_CREDENTIALS"] = "../../configs/zihan_gcs_token.json"
os.environ["AWS_EC2_METADATA_DISABLED"] = "true"

PROJ = Path("../..").resolve()                      # exa-spim-agent/
sys.path.insert(0, str(PROJ))
sys.path.insert(0, str(PROJ.parent / "segmentation-skeleton-metrics" / "src"))
sys.path.insert(0, str(PROJ / "scripts"))

from proofreader_evolve.harness import incremental_scoring as inc
from proofreader_evolve.harness import dataset as ds
from proofreader_evolve.harness import candidate as cand
from proofreader_evolve.harness import scoring
from proofreader_evolve.harness.edit_handler import normalize_edits

# --- Parameters -----------------------------------------------------------
RUN_NAME = "789202_train1_test1_20260628_212929"   # the cross-brain evolution run id
GEN = None                                      # None -> last ACCEPTED generation (best policy)

# Which brain group to evaluate THIS kernel session. ONE group per run keeps only a
# single large fragment graph in memory at a time. To do both: run "test", then
# RESTART THE KERNEL, flip to "train", and run again.
TRAIN_OR_TEST = "test"                           # "test" (held-out) | "train" (sanity check)

RUNS = PROJ / "proofreader_evolve" / "runs"
RUN_DIR = RUNS / RUN_NAME
assert RUN_DIR.is_dir(), f"run dir not found: {RUN_DIR}"

TRAIN_OR_TEST = str(TRAIN_OR_TEST).strip().lower()
assert TRAIN_OR_TEST in ("train", "test"), \
    f"TRAIN_OR_TEST must be 'train' or 'test', got {TRAIN_OR_TEST!r}"

# Auto-resolve GEN to the last accepted generation from the ledger.
_ledger = [json.loads(l) for l in (RUN_DIR / "ledger.jsonl").read_text().splitlines() if l.strip()]
if GEN is None:
    _accepted = [r["generation"] for r in _ledger if r.get("accepted")]
    assert _accepted, f"no accepted generation in {RUN_DIR/'ledger.jsonl'}"
    GEN = f"gen{max(_accepted):02d}"
    print(f"GEN auto-resolved to last accepted generation: {GEN} (accepted: {sorted(_accepted)})")

# splits_only mode of the run (drives whether MergeSites are enumerated).
SPLITS_ONLY = bool(_ledger[0].get("splits_only", False))

# Read the split config: cross-brain test/train brains, mcl, per-brain partition.
_split = json.loads((RUN_DIR / "split.json").read_text())
MCL = int(_split.get("mcl", 100))
CROSS_BRAIN = bool(_split.get("cross_brain", False))
if CROSS_BRAIN:
    TEST_BRAINS = [str(b) for b in (_split.get("test_brains") or [])]
    TRAIN_BRAINS = [str(b) for b in (_split.get("train_brains") or [])]
    print(f"CROSS-BRAIN run: test brains = {TEST_BRAINS}; train brains = {TRAIN_BRAINS}")
else:
    # Not a cross-brain run: fall back to every brain's held-out SUBSET (per-brain split),
    # and use the SAME brains' TRAIN subset for the sanity check.
    TEST_BRAINS = list(_split.get("per_brain", {}).keys())
    TRAIN_BRAINS = list(_split.get("per_brain", {}).keys())
    print(f"[WARN] run is NOT cross-brain (no --test-brains). Falling back to the per-brain "
          f"subset of: {TEST_BRAINS}. Numbers are the within-brain held-out/train, not a "
          f"whole-brain set.")

# Select exactly ONE group to load + score this session.
if TRAIN_OR_TEST == "test":
    BRAINS, SUBSET, ROLE = TEST_BRAINS, "heldout", "TEST"
else:
    BRAINS, SUBSET, ROLE = TRAIN_BRAINS, "train", "TRAIN"
assert BRAINS, f"no {ROLE} brains found in split.json for TRAIN_OR_TEST={TRAIN_OR_TEST!r}"

POLICY = RUN_DIR / GEN / "heuristics.accepted.py"
assert POLICY.exists(), f"accepted policy not found: {POLICY}"
print(f"policy: {POLICY}")
print(f"mcl={MCL}  splits_only={SPLITS_ONLY}  generations in ledger={len(_ledger)}")
print(f"\nTHIS RUN: TRAIN_OR_TEST={TRAIN_OR_TEST!r} -> evaluating {ROLE} brains {BRAINS} "
      f"on their '{SUBSET}' neurons.")
print("(one group per kernel session — restart the kernel and flip TRAIN_OR_TEST for the other.)")

GEN auto-resolved to last accepted generation: gen39 (accepted: [1, 2, 5, 7, 9, 14, 15, 18, 19, 21, 23, 28, 30, 31, 39])
CROSS-BRAIN run: test brains = ['794491']; train brains = ['789202']
policy: /allen/programs/mindscope/workgroups/auto-model/zihan.zhang/exaspim-agent/exa-spim-agent/proofreader_evolve/runs/789202_train1_test1_20260628_212929/gen39/heuristics.accepted.py
mcl=10  splits_only=True  generations in ledger=40

THIS RUN: TRAIN_OR_TEST='test' -> evaluating TEST brains ['794491'] on their 'heldout' neurons.
(one group per kernel session — restart the kernel and flip TRAIN_OR_TEST for the other.)


## 1. Load the policy once (shared across the selected brains)

In [2]:
propose_edits, policy_module = cand._load_policy(str(POLICY))
raw_enum = getattr(policy_module, "ENUM_PARAMS", None)
raw_enum = {**raw_enum} if isinstance(raw_enum, dict) else {}
raw_enum.setdefault("max_gap_um", 15.0)
ENUM_PARAMS = ds.resolve_enum_params(raw_enum)
print("resolved ENUM_PARAMS:", ENUM_PARAMS)

resolved ENUM_PARAMS: {'max_gap_um': 15.0, 'split_max_sites': 25000, 'tip_to_shaft': True, 'split_alt_per_pair': 1, 'min_arm_cable_um': 10.0, 'seed_depth_um': 8.0, 'merge_max_sites': 5000, 'max_per_label': 8}


## 2. Per-brain evaluation function

For one brain: load its prepared state + the matching fragment cache, run the
policy (with the SAME image reader the loop used), FREE the large fragment graph,
then score baseline-vs-edited on the chosen neuron `subset` and classify the merge
edits against that subset's label→neuron map.

`subset="heldout"` scores the held-out neurons — for a cross-brain TEST brain that
is the whole brain, a pure generalization measurement. `subset="train"` scores the
TRAIN neurons the reviser got feedback on — a SANITY CHECK (high repair expected).

The returned dict keeps only lightweight results (scores + counts + coverage), not
the multi-MB site/edit lists or the graph, so memory stays flat across brains.

In [3]:
def _pair_key(a, b):
    a, b = str(a), str(b)
    return (a, b) if a <= b else (b, a)

def analyze_brain(brain_id, subset="heldout"):
    '''Run the policy on `brain_id`, scoring on the chosen neuron `subset`.

    subset="heldout" -> the held-out neurons (the true generalization signal; used
                        for the cross-brain TEST brains).
    subset="train"   -> the TRAIN neurons the reviser got feedback on (a SANITY
                        CHECK: high split-repair here is EXPECTED, not generalization).
    Returns lightweight measurements as a dict (heavy graph/site/edit objects are
    freed before returning so only one brain's footprint is live at a time).
    '''
    assert subset in ("heldout", "train"), subset
    bdir_prepared = RUN_DIR / f"prepared_{brain_id}.pkl"
    assert bdir_prepared.exists(), (
        f"prepared cache missing for brain {brain_id}: {bdir_prepared} "
        f"(the run must have set this brain up; check it is in split.json per_brain)")
    prepared = inc.PreparedBrain.load(str(bdir_prepared))

    # SAME fragment cache the run enumerated over (validated against brain+mcl).
    cache_path = ds.default_cache_path(brain_id, min_cable_length=MCL)
    fragments_graph, _gt, _payload = ds.load_cached_graphs(
        cache_path, expect_brain=brain_id, expect_mcl=MCL)

    # This brain's neuron partition from split.json. Score on the requested subset:
    # for a cross-brain TEST brain heldout == all neurons (train == []); for a TRAIN
    # brain the train subset is every neuron (heldout == []).
    pb = _split["per_brain"][brain_id]
    heldout_names = sorted(pb["heldout"])
    train_names = sorted(pb["train"])
    score_names = heldout_names if subset == "heldout" else train_names
    assert score_names, (
        f"brain {brain_id} has no '{subset}' neurons in split.json "
        f"(heldout={len(heldout_names)}, train={len(train_names)})")

    # IMAGE READER — must match the evolution loop (gen policy gates recall on it).
    try:
        from proofreader_evolve.harness.image_features import LazyImagePatchReader
        from dataset_config import get_img_path
        _prefixes = str(PROJ / "configs" / "exaspim_image_prefixes.json")
        _img_path = get_img_path(brain_id, prefixes_path=_prefixes)
        image_reader = LazyImagePatchReader(_img_path, fragments_graph)
        image_on = True
    except Exception as _e:
        image_reader, image_on = None, False
        print(f"[WARN] {brain_id}: could not build image reader ({_e}); image tiers "
              f"DISABLED -> edits are the geometry-only subset, NOT the true policy.")

    # Enumerate the SAME candidate stream (merges only if the run scored merges).
    # _enumerate_sites_cached returns (split_sites, merge_sites, split_enum_stats);
    # the trailing stats dict is the GT-free enumeration-ceiling info (unused here).
    split_sites, merge_sites, _split_stats = cand._enumerate_sites_cached(
        fragments_graph, ENUM_PARAMS, enumerate_merges=not SPLITS_ONLY)
    sites = list(split_sites) + list(merge_sites)
    ctx = {
        "max_gap_um": ENUM_PARAMS["max_gap_um"], "enum_params": dict(ENUM_PARAMS),
        "fragments_graph": fragments_graph, "n_split_sites": len(split_sites),
        "n_merge_sites": len(merge_sites),
        "node_radius": getattr(fragments_graph, "node_radius", None),
        "read_image_patch": image_reader,
    }
    edits = normalize_edits(propose_edits(sites, ctx))
    if SPLITS_ONLY:
        edits = [e for e in edits if e.get("kind") == "merge_labels"]
    n_split_sites, n_edits = len(split_sites), len(edits)

    # Split-repair classification against the chosen subset's label->neuron map.
    score_map = inc.label_gt_counts(prepared, gt_names=score_names)
    repair = inc.classify_merge_edits(edits, score_map)

    # PER-SITE coverage: distinct reachable real splits fixed vs remaining (this subset).
    accepted = {_pair_key(e.get("label_a"), e.get("label_b"))
                for e in edits if e.get("kind") == "merge_labels" and e.get("label_a")}
    def _dom(lbl):
        c = score_map.get(str(lbl)); return max(c, key=c.get) if c else None
    fixed = remaining = 0
    for s in split_sites:
        la, lb = str(s.label_a), str(s.label_b)
        da, db = _dom(la), _dom(lb)
        if da is not None and da == db:            # reachable real split on a scored neuron
            if _pair_key(la, lb) in accepted: fixed += 1
            else: remaining += 1

    # Free the large fragment graph + enumerated sites BEFORE scoring (scoring uses
    # only `prepared`), so peak memory holds one graph, not graph + score frames.
    del sites, split_sites, merge_sites, ctx, fragments_graph, image_reader, accepted, score_map
    gc.collect()

    # Score baseline vs edited, RESTRICTED to the chosen neuron subset.
    base = inc.score_incremental(prepared, label_pairs=None, gt_swc_names=score_names, verbose=False)
    edited = inc.score_incremental(prepared, edits=edits, gt_swc_names=score_names, verbose=False)

    return {
        "brain": brain_id, "subset": subset, "image_on": image_on,
        "n_scored": len(score_names), "n_heldout": len(heldout_names), "n_train": len(train_names),
        "n_split_sites": n_split_sites, "n_edits": n_edits,
        "base": base, "edited": edited, "repair": repair,
        "coverage": {"fixed": fixed, "remaining": remaining, "reachable": fixed + remaining},
    }

## 3. Run the selected brain group (ONE group per kernel session)

In [4]:
results = {}
print(f"########## {ROLE} brains — scoring on '{SUBSET}' neurons (ONE group this kernel run) ##########")
if ROLE == "TRAIN":
    print("########## SANITY CHECK: the policy was evolved against these neurons, so high")
    print("########## split-repair coverage is EXPECTED — NOT a generalization claim. ##########")

for b in BRAINS:
    print()
    print(f"=== analyzing {ROLE} brain {b} (scored on '{SUBSET}' neurons) ===")
    results[b] = analyze_brain(b, subset=SUBSET)
    r = results[b]
    img = "image ON" if r["image_on"] else "GEOMETRY-ONLY (image off)"
    print(f"  {b}: {r['n_scored']} {SUBSET} neurons, {r['n_split_sites']} candidate pairs, "
          f"{r['n_edits']} merge edits  [{img}]")
    gc.collect()   # release this brain before loading the next (if more than one)

########## TEST brains — scoring on 'heldout' neurons (ONE group this kernel run) ##########

=== analyzing TEST brain 794491 (scored on 'heldout' neurons) ===
  794491: 9 heldout neurons, 25000 candidate pairs, 8656 merge edits  [image ON]


## 4. Per-skeleton metric comparison (baseline vs edited)

Every row is a scored neuron of the selected group (TEST: held-out neurons; TRAIN:
the train neurons the policy saw). `d…` columns are edited − baseline; `edit-caused
merges` is how many cross-neuron fusions the policy INTRODUCED on that neuron (the
cause behind `d% Merged Edges`, which can net to ~0 when new merges cancel
pre-existing ones).

In [5]:
COLS = ["Edge Accuracy", "% Merged Edges", "# Merges", "% Split Edges", "# Splits", "% Omit Edges"]

def _edit_caused(edited, index):
    ms = getattr(edited, "merge_sites", None)
    if (ms is None or len(ms) == 0 or "GroundTruth_ID" not in getattr(ms, "columns", [])
            or "Caused_By_Edit" not in ms.columns):
        return pd.Series(0, index=index, dtype=int)
    rows = ms[ms["Caused_By_Edit"]]
    return rows["GroundTruth_ID"].value_counts().reindex(index).fillna(0).astype(int)

def comparison_table(r):
    base, edited = r["base"], r["edited"]
    b = base.per_swc[COLS]
    e = edited.per_swc.reindex(b.index)[COLS]
    delta = (e - b).add_prefix("d")
    comp = pd.concat([b.add_prefix("base "), e.add_prefix("edit "), delta], axis=1)
    comp["edit-caused merges"] = _edit_caused(edited, b.index)
    comp.loc["WEIGHTED MEAN"] = {
        **{f"base {c}": scoring._weighted_avg(base.per_swc, c) for c in COLS},
        **{f"edit {c}": scoring._weighted_avg(edited.per_swc, c) for c in COLS},
        **{f"d{c}": scoring._weighted_avg(edited.per_swc, c) - scoring._weighted_avg(base.per_swc, c) for c in COLS},
    }
    comp.loc["WEIGHTED MEAN", "edit-caused merges"] = int(comp["edit-caused merges"][:-1].sum())
    return comp

pd.set_option("display.width", 220, "display.max_columns", 40)
for b in BRAINS:
    if b not in results:
        continue
    print(f"################  {ROLE} BRAIN {b}  ################")
    display(comparison_table(results[b]).round(3))

################  TEST BRAIN 794491  ################


,base Edge Accuracy,base % Merged Edges,base # Merges,base % Split Edges,base # Splits,base % Omit Edges,edit Edge Accuracy,edit % Merged Edges,edit # Merges,edit % Split Edges,edit # Splits,edit % Omit Edges,dEdge Accuracy,d% Merged Edges,d# Merges,d% Split Edges,d# Splits,d% Omit Edges,edit-caused merges
N001-794491-JG,68.940,29.949,13.000,0.423,801.000,0.689,68.890,30.000,13.000,0.422,798.00,0.689,-0.050,0.052,0.000,-0.001,-3.000,0.000,5.0
N002-794491-PP,72.550,25.029,29.000,0.677,1070.000,1.739,72.350,25.242,30.000,0.670,1058.00,1.737,-0.200,0.213,1.000,-0.007,-12.000,-0.003,10.0
N004-794491-MB,69.380,27.684,21.000,0.563,628.000,2.373,63.080,33.989,23.000,0.558,622.00,2.370,-6.300,6.305,2.000,-0.005,-6.000,-0.004,10.0
N005-794491-HP,46.660,52.368,3.000,0.330,398.000,0.637,46.670,52.368,3.000,0.327,397.00,0.629,0.010,0.000,0.000,-0.003,-1.000,-0.008,0.0
N006-794491-SP,96.130,3.341,4.000,0.313,471.000,0.219,96.140,3.341,4.000,0.310,468.00,0.214,0.010,0.000,0.000,-0.003,-3.000,-0.004,1.0
N008-794491-SP,68.490,27.630,37.000,0.911,1617.000,2.965,67.130,28.995,37.000,0.906,1608.00,2.965,-1.360,1.365,0.000,-0.006,-9.000,0.000,17.0
N009-794491-JG,80.880,10.130,9.000,1.468,1541.000,7.526,79.960,11.048,10.000,1.466,1536.00,7.526,-0.920,0.918,1.000,-0.002,-5.000,0.000,1.0
N012-794491-SP,82.230,14.536,17.000,0.868,1738.000,2.364,81.920,14.865,18.000,0.862,1725.00,2.355,-0.310,0.329,1.000,-0.006,-13.000,-0.009,2.0
N013-794491-PP,81.310,7.488,15.000,1.755,3104.000,9.443,81.260,7.564,15.000,1.745,3083.00,9.427,-0.050,0.076,0.000,-0.009,-21.000,-0.017,1.0
WEIGHTED MEAN,75.014,21.102,17.332,0.824,1354.763,3.059,74.217,21.910,17.835,0.819,1345.82,3.054,-0.797,0.808,0.503,-0.005,-8.944,-0.005,47.0


## 5. Split-repair coverage for the selected group

TEST numbers are the headline held-out generalization result. TRAIN numbers are a
SANITY CHECK: the policy was evolved against these neurons, so high coverage is
EXPECTED. Run both groups (restart the kernel between them) and read the GAP between
TRAIN and TEST coverage to gauge how much of the policy's repair power transfers vs.
is train-specific.

In [6]:
def _pct(n, d): return f"{100.0*n/d:.1f}%" if d else "--"

def coverage_summary(results_dict, brains, role):
    rows = []
    for b in brains:
        if b not in results_dict:
            continue
        r = results_dict[b]; rep = r["repair"]; cov = r["coverage"]
        img = "" if r["image_on"] else "   [GEOMETRY-ONLY — image off, NOT the true score]"
        scope = "whole-brain held-out" if r["subset"] == "heldout" else "TRAIN neurons (sanity)"
        print()
        print(f"=== {role} BRAIN {b} ({scope}){img} ===")
        print(f"  PER-EDIT split-repair (gate signal): correct={rep['correct']}  "
              f"false={rep['false']}  unscored={rep['unscored']}  "
              f"score(correct-false)={rep['correct']-rep['false']}")
        print(f"  PER-SITE coverage: {cov['fixed']} of {cov['reachable']} distinct reachable "
              f"split errors fixed ({_pct(cov['fixed'], cov['reachable'])}); "
              f"{cov['remaining']} reachable-but-unrepaired")
        base_acc = scoring._weighted_avg(r["base"].per_swc, "Edge Accuracy")
        edit_acc = scoring._weighted_avg(r["edited"].per_swc, "Edge Accuracy")
        base_mrg = scoring._weighted_avg(r["base"].per_swc, "% Merged Edges")
        edit_mrg = scoring._weighted_avg(r["edited"].per_swc, "% Merged Edges")
        print(f"  Edge Accuracy {base_acc:.3f} -> {edit_acc:.3f} ({edit_acc-base_acc:+.3f}); "
              f"% Merged Edges {base_mrg:.3f} -> {edit_mrg:.3f} ({edit_mrg-base_mrg:+.3f})")
        rows.append({
            "brain": b, "role": role, "subset": r["subset"],
            "image_on": r["image_on"], "n_scored": r["n_scored"],
            "correct": rep["correct"], "false": rep["false"], "unscored": rep["unscored"],
            "split_repair_score": rep["correct"] - rep["false"],
            "reachable": cov["reachable"], "fixed": cov["fixed"], "remaining": cov["remaining"],
            "coverage_pct": (100.0*cov["fixed"]/cov["reachable"] if cov["reachable"] else float("nan")),
            "base_edge_acc": base_acc, "edit_edge_acc": edit_acc,
            "base_pct_merged": base_mrg, "edit_pct_merged": edit_mrg,
        })
    return rows

if ROLE == "TEST":
    print("########## TEST (held-out) — the headline generalization number ##########")
else:
    print("########## TRAIN (sanity check — high coverage EXPECTED, NOT generalization) ##########")
_rows = coverage_summary(results, BRAINS, ROLE)

summary = pd.DataFrame(_rows).set_index(["role", "brain"])
print()
print(f"=== {ROLE} summary ===")
display(summary.round(3))

########## TEST (held-out) — the headline generalization number ##########

=== TEST BRAIN 794491 (whole-brain held-out) ===
  PER-EDIT split-repair (gate signal): correct=73  false=0  unscored=8583  score(correct-false)=73
  PER-SITE coverage: 73 of 203 distinct reachable split errors fixed (36.0%); 130 reachable-but-unrepaired
  Edge Accuracy 75.014 -> 74.217 (-0.797); % Merged Edges 21.102 -> 21.910 (+0.808)

=== TEST summary ===


,,subset,image_on,n_scored,correct,false,unscored,split_repair_score,reachable,fixed,remaining,coverage_pct,base_edge_acc,edit_edge_acc,base_pct_merged,edit_pct_merged
role,brain,,,,,,,,,,,,,,,
TEST,794491,heldout,True,9,73,0,8583,73,203,73,130,35.961,75.014,74.217,21.102,21.91


## 6. Save the selected group's analysis to the run folder

In [7]:
ana_dir = RUN_DIR / "test_brain_analysis"
ana_dir.mkdir(parents=True, exist_ok=True)

# Summary CSV for THIS group (suffixed by group so the test/train runs don't clobber).
summary_csv = ana_dir / f"brain_summary_{GEN}_{TRAIN_OR_TEST}.csv"
summary.to_csv(summary_csv)

for b in BRAINS:
    if b not in results:
        continue
    r = results[b]; rep = r["repair"]; cov = r["coverage"]
    payload = {
        "run": RUN_NAME, "generation": GEN, "brain": b, "role": TRAIN_OR_TEST, "subset": r["subset"],
        "cross_brain": CROSS_BRAIN, "splits_only": SPLITS_ONLY, "mcl": MCL,
        "image_reader_enabled": r["image_on"],
        "n_scored_neurons": r["n_scored"], "n_distinct_candidate_pairs": r["n_split_sites"],
        "n_merge_labels_edits": r["n_edits"],
        "per_edit": {k: rep[k] for k in ("correct", "false", "unscored")},
        "per_site": cov,
    }
    (ana_dir / f"{TRAIN_OR_TEST}_brain_coverage_{GEN}_{b}.json").write_text(json.dumps(payload, indent=2))
    comparison_table(r).to_csv(ana_dir / f"{TRAIN_OR_TEST}_brain_per_skeleton_{GEN}_{b}.csv")

print(f"saved {ROLE} summary -> {summary_csv}")
print(f"saved per-brain JSON + per-skeleton CSV ({TRAIN_OR_TEST}_*) under -> {ana_dir}")
if not all(results[b]["image_on"] for b in BRAINS if b in results):
    print("[WARN] one or more brains scored with image OFF — those rows are the "
          "geometry-only subset, not the policy's true behaviour. Re-run with GCS access.")
print(f"\nTo evaluate the other group: RESTART THE KERNEL, set "
      f"TRAIN_OR_TEST = {'train' if TRAIN_OR_TEST=='test' else 'test'!r} in cell 0, and re-run.")

saved TEST summary -> /allen/programs/mindscope/workgroups/auto-model/zihan.zhang/exaspim-agent/exa-spim-agent/proofreader_evolve/runs/789202_train1_test1_20260628_212929/test_brain_analysis/brain_summary_gen39_test.csv
saved per-brain JSON + per-skeleton CSV (test_*) under -> /allen/programs/mindscope/workgroups/auto-model/zihan.zhang/exaspim-agent/exa-spim-agent/proofreader_evolve/runs/789202_train1_test1_20260628_212929/test_brain_analysis

To evaluate the other group: RESTART THE KERNEL, set TRAIN_OR_TEST = 'train' in cell 0, and re-run.


## 7. Diagnostic: GATE verdict vs. the `# Merges` metric (why they can disagree)

The gate and the `# Merges` table column are **two different measurements** and can
legitimately disagree (e.g. `# Merges` rises while the gate sees zero false merges):

* **GATE** — `classify_merge_edits`: for each `merge_labels` edit it compares the
  **argmax (dominant) GT neuron** of the two fused labels. `false` iff they differ.
  No geometry, no node-count threshold. This — `correct - false`, reject if any
  held-out `false` — is the ONLY signal the accept/reject gate uses.
* **`# Merges`** — `MergeCountMetric`: a **geometric** count. It walks fragment
  leaves, flags one that sits `>50µm` from its own GT skeleton and then `<6µm` from
  ANOTHER, and counts that as a merge SITE. It never looks at argmax.

So an argmax-CORRECT merge (both labels mostly belong to neuron N1) can still drag a
physical arm into a spatial collision with a neighbouring GT skeleton — a geometric
site `# Merges` counts but the gate does not. This cell shows both, side by side, for
one brain + neuron, so you can confirm the mechanism empirically.

Uses only the light-weight `results` dict (`repair` = gate pairs; `edited.merge_sites`
= geometric sites with `Caused_By_Edit`), so no brain reload is needed.

In [ ]:
# --- Parameters for the diagnostic ---------------------------------------
DIAG_BRAIN = BRAINS[0]      # which scored brain to inspect
DIAG_NEURON = None          # None -> auto-pick the neuron with the most edit-caused merge SITES

assert DIAG_BRAIN in results, f"{DIAG_BRAIN} not in results (run cell 3 first)"
r = results[DIAG_BRAIN]
rep = r["repair"]
edited = r["edited"]
ms = getattr(edited, "merge_sites", None)
has_ms = ms is not None and len(ms) and "Caused_By_Edit" in getattr(ms, "columns", [])

print(f"================  {ROLE} brain {DIAG_BRAIN}: GATE  vs  '# Merges' metric  ================\n")

# (1) GATE measurement — argmax-based, brain-wide, the ONLY signal the gate uses.
print("(1) GATE  [classify_merge_edits — argmax of each fused label, NO geometry]")
print(f"      correct={rep['correct']}  false={rep['false']}  unscored={rep['unscored']}")
print(f"      gate fitness = correct - false = {rep['correct'] - rep['false']}")
print(f"      a generation is REJECTED iff false > 0 on HELD-OUT; here false={rep['false']}.")
if rep.get("false_pairs"):
    print("      gate-FALSE merge edits (fused two DIFFERENT dominant neurons):")
    display(pd.DataFrame(rep["false_pairs"],
                         columns=["label_a", "label_b", "neuron_a", "neuron_b"]))
else:
    print("      gate-FALSE merge edits: NONE — every merge edit was argmax-correct.")

# (2) '# Merges' metric measurement — geometric merge SITES the policy introduced.
print("\n(2) '# Merges'  [MergeCountMetric — geometric SITES: a fragment leaf >50um from")
print("      its own GT skeleton, then <6um to ANOTHER GT skeleton]")
if not has_ms:
    print("      no edit-caused merge_sites attributed on the scored neurons.")
else:
    caused = ms[ms["Caused_By_Edit"]]
    print(f"      total edit-caused merge SITES across scored neurons: {len(caused)}")
    if len(caused):
        display(caused["GroundTruth_ID"].value_counts().to_frame("edit_caused_sites"))

# --- Per-neuron drill-down ------------------------------------------------
if DIAG_NEURON is None and has_ms:
    _caused = ms[ms["Caused_By_Edit"]]
    if len(_caused):
        DIAG_NEURON = _caused["GroundTruth_ID"].value_counts().idxmax()

if DIAG_NEURON is not None:
    print(f"\n----------------  drill-down: neuron {DIAG_NEURON}  ----------------")
    # The geometric sites '# Merges' / '% Merged Edges' react to on this neuron:
    if has_ms:
        rows = ms[(ms["GroundTruth_ID"] == DIAG_NEURON) & (ms["Caused_By_Edit"])]
        print(f"  '# Merges' view: {len(rows)} edit-caused merge SITE(s) on {DIAG_NEURON}")
        if len(rows):
            cols = [c for c in ["Label", "Fused_Labels", "GroundTruth_ID", "World", "Voxel"]
                    if c in rows.columns]
            display(rows[cols].reset_index(drop=True))
    # Any gate-FALSE pairs that touch this neuron (usually none on a clean held-out gate):
    touching = [p for p in rep.get("false_pairs", [])
                if str(DIAG_NEURON) in (str(p[2]), str(p[3]))]
    print(f"  GATE view: {len(touching)} gate-FALSE merge edit(s) touching {DIAG_NEURON} "
          f"(argmax of the two fused labels disagree)")
    if touching:
        display(pd.DataFrame(touching, columns=["label_a", "label_b", "neuron_a", "neuron_b"]))

    print("\n  INTERPRETATION: edit-caused merge SITES can be > 0 while the gate's FALSE "
          "count is 0 — the\n  gate scores the ARGMAX of each fused label (no geometry), "
          "whereas '# Merges' counts a\n  geometric collision (a fused arm landing on a "
          "NEIGHBOURING GT skeleton). An argmax-correct\n  split repair can still drag an "
          "arm into such a collision, so the two measurements diverge.")
else:
    print("\n(no neuron had an edit-caused merge site; set DIAG_NEURON manually to inspect one.)")